In [1]:

# ==========================================
# PHASE 5: PDC CALCULATION
# ==========================================

import pandas as pd
import numpy as np

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from getpass import getpass

# Connect to the existing MySQL database

db_password = getpass("Enter your MySQL password: ")

connection_url = URL.create(
    drivername="mysql+mysqlconnector",
    username="root",
    password=db_password,
    host="localhost",
    port=3306,
    database="medication_adherence"
)

engine = create_engine(
    connection_url,
    pool_pre_ping=True
)

print("MySQL connection established!")


Enter your MySQL password:  ········


MySQL connection established!


In [7]:

# Reload cohort refill data, including drug_name

cohort_refills = pd.read_sql_query(
    """
    SELECT
        fill_id,
        patient_id,
        measure,
        drug_name,
        filled_on,
        days_supply,
        measurement_start,
        measurement_end,
        measurement_days

    FROM cohort_refills

    ORDER BY
        patient_id,
        measure,
        filled_on,
        fill_id
    """,
    con=engine
)

# Convert date columns

date_columns = [
    "filled_on",
    "measurement_start",
    "measurement_end"
]

for col in date_columns:
    cohort_refills[col] = pd.to_datetime(
        cohort_refills[col],
        errors="raise"
    )

print("Cohort refill data reloaded successfully!")
print("Total records:", len(cohort_refills))
print("Drug name column available:", "drug_name" in cohort_refills.columns)


Cohort refill data reloaded successfully!
Total records: 24640
Drug name column available: True


In [8]:

# ==========================================
# TEST: REFILL OVERLAP AND CARRY-FORWARD
# ==========================================

import pandas as pd

# Create artificial test data (not real patient data)

test_refills = pd.DataFrame({
    "fill_id": [1, 2],

    "filled_on": pd.to_datetime([
        "2025-01-01",
        "2025-01-20"
    ]),

    "days_supply": [30, 30],

    "measurement_start": pd.to_datetime([
        "2025-01-01",
        "2025-01-01"
    ]),

    "measurement_end": pd.to_datetime([
        "2025-03-31",
        "2025-03-31"
    ])
})


# Function to calculate carry-forward coverage

def calculate_refill_coverage(group):

    group = group.sort_values(
        ["filled_on", "fill_id"]
    ).copy()

    previous_effective_end = None

    results = []

    for row in group.itertuples(index=False):

        fill_date = row.filled_on.normalize()

        window_start = row.measurement_start.normalize()

        window_end = row.measurement_end.normalize()

        # Determine effective start date

        if previous_effective_end is None:

            effective_start = fill_date

        else:

            effective_start = max(
                fill_date,
                previous_effective_end + pd.Timedelta(days=1)
            )

        # Calculate effective end date

        effective_end = (
            effective_start
            + pd.Timedelta(days=int(row.days_supply) - 1)
        )

        # Limit coverage to measurement window

        covered_start = max(
            effective_start,
            window_start
        )

        covered_end = min(
            effective_end,
            window_end
        )

        # Calculate inclusive covered days

        covered_days = max(
            0,
            (covered_end - covered_start).days + 1
        )

        results.append({
            "effective_start": effective_start,
            "effective_end": effective_end,
            "covered_days": covered_days
        })

        # Preserve unused supply for subsequent refills

        previous_effective_end = effective_end

    return pd.DataFrame(results, index=group.index)


# Run the synthetic test

test_result = calculate_refill_coverage(test_refills)

display(test_result)

print(
    "Total covered days:",
    test_result["covered_days"].sum()
)


,effective_start,effective_end,covered_days
0,2025-01-01,2025-01-30,30
1,2025-01-31,2025-03-01,30


Total covered days: 60


In [9]:

# ==========================================
# APPLY REFILL CARRY-FORWARD ALGORITHM
# ==========================================

def calculate_refill_coverage(group):

    # Sort prescriptions chronologically
    group = group.sort_values(
        ["filled_on", "fill_id"]
    ).copy()

    previous_effective_end = None

    effective_starts = []
    effective_ends = []
    covered_days_list = []

    for row in group.itertuples(index=False):

        fill_date = pd.Timestamp(row.filled_on).normalize()

        window_start = pd.Timestamp(
            row.measurement_start
        ).normalize()

        window_end = pd.Timestamp(
            row.measurement_end
        ).normalize()

        # Calculate effective refill start
        if previous_effective_end is None:

            effective_start = fill_date

        else:

            effective_start = max(
                fill_date,
                previous_effective_end + pd.Timedelta(days=1)
            )

        # Calculate full effective refill end
        effective_end = (
            effective_start
            + pd.Timedelta(days=int(row.days_supply) - 1)
        )

        # Restrict covered dates to measurement window
        covered_start = max(effective_start, window_start)

        covered_end = min(effective_end, window_end)

        # Calculate inclusive covered days
        covered_days = max(
            0,
            (covered_end - covered_start).days + 1
        )

        effective_starts.append(effective_start)
        effective_ends.append(effective_end)
        covered_days_list.append(covered_days)

        # Carry forward unused medication supply
        previous_effective_end = effective_end

    # Add calculated columns to the original group
    group["effective_start"] = effective_starts
    group["effective_end"] = effective_ends
    group["covered_days"] = covered_days_list

    return group


# Process each patient-measure group separately

processed_groups = []

for _, group in cohort_refills.groupby(
    ["patient_id", "measure"],
    sort=False
):

    processed_groups.append(
        calculate_refill_coverage(group)
    )


# Combine processed groups

covered_refills = pd.concat(
    processed_groups,
    ignore_index=True
)

print("REFILL OVERLAP CALCULATION COMPLETED!")
print("Total refill records:", len(covered_refills))

print(
    "Patient-measure groups:",
    covered_refills.groupby(
        ["patient_id", "measure"]
    ).ngroups
)


REFILL OVERLAP CALCULATION COMPLETED!
Total refill records: 24640
Patient-measure groups: 3565


In [10]:

# ==========================================
# VALIDATE REFILL COVERAGE
# ==========================================

# Summarize covered days for each patient and measure

patient_coverage_check = (
    covered_refills
    .groupby(
        ["patient_id", "measure"],
        as_index=False
    )
    .agg(
        total_covered_days=("covered_days", "sum"),
        measurement_days=("measurement_days", "first")
    )
)

# Display aggregate validation

print("COVERAGE VALIDATION")
print("=" * 45)

print("Total refill records:", len(covered_refills))

print(
    "Unique prescription IDs:",
    covered_refills["fill_id"].nunique()
)

print(
    "Patient-measure groups:",
    len(patient_coverage_check)
)

print(
    "Negative covered-day records:",
    (covered_refills["covered_days"] < 0).sum()
)

print(
    "Refills exceeding their days supply:",
    (
        covered_refills["covered_days"]
        > covered_refills["days_supply"]
    ).sum()
)

print(
    "Groups exceeding measurement period:",
    (
        patient_coverage_check["total_covered_days"]
        > patient_coverage_check["measurement_days"]
    ).sum()
)


COVERAGE VALIDATION
Total refill records: 24640
Unique prescription IDs: 24640
Patient-measure groups: 3565
Negative covered-day records: 0
Refills exceeding their days supply: 0
Groups exceeding measurement period: 0


In [11]:

# ==========================================
# AGGREGATE COVERED DAYS PER PATIENT
# ==========================================

patient_adherence = (
    covered_refills
    .groupby(
        ["patient_id", "measure"],
        as_index=False
    )
    .agg(
        counting_refills=("fill_id", "nunique"),

        distinct_drugs=("drug_name", "nunique"),

        measurement_start=("measurement_start", "first"),

        measurement_end=("measurement_end", "first"),

        measurement_days=("measurement_days", "first"),

        total_covered_days=("covered_days", "sum")
    )
)

print("PATIENT COVERAGE AGGREGATION COMPLETED!")

print(
    "Patient-measure groups:",
    len(patient_adherence)
)

print(
    "Unique patients:",
    patient_adherence["patient_id"].nunique()
)


PATIENT COVERAGE AGGREGATION COMPLETED!
Patient-measure groups: 3565
Unique patients: 2181


In [12]:

# ==========================================
# CALCULATE PROPORTION OF DAYS COVERED
# ==========================================

# Calculate PDC for each patient and measure

patient_adherence["pdc"] = (
    patient_adherence["total_covered_days"]
    / patient_adherence["measurement_days"]
)

# Convert PDC into percentage

patient_adherence["pdc_percentage"] = (
    patient_adherence["pdc"] * 100
)

print("PDC CALCULATION COMPLETED!")
print("Patient-measure groups:", len(patient_adherence))
print("Adherence threshold: 80%")


PDC CALCULATION COMPLETED!
Patient-measure groups: 3565
Adherence threshold: 80%


In [13]:

# ==========================================
# ADHERENCE CLASSIFICATION
# ==========================================

# Classify patients using the 80% threshold

patient_adherence["adherent"] = (
    patient_adherence["pdc"] >= 0.80
)

# Assign readable adherence labels

patient_adherence["adherence_status"] = np.where(
    patient_adherence["adherent"],
    "Adherent",
    "Non-Adherent"
)

print("ADHERENCE CLASSIFICATION COMPLETED!")

print(
    "Adherent groups:",
    patient_adherence["adherent"].sum()
)

print(
    "Non-adherent groups:",
    (~patient_adherence["adherent"]).sum()
)


ADHERENCE CLASSIFICATION COMPLETED!
Adherent groups: 2768
Non-adherent groups: 797


In [14]:

# ==========================================
# PDC VALIDATION
# ==========================================

print("PDC VALIDATION SUMMARY")
print("=" * 45)

print(
    "Total patient-measure groups:",
    len(patient_adherence)
)

print(
    "Missing PDC values:",
    patient_adherence["pdc"].isna().sum()
)

print(
    "PDC below zero:",
    (patient_adherence["pdc"] < 0).sum()
)

print(
    "PDC above 1:",
    (patient_adherence["pdc"] > 1).sum()
)

print(
    "Minimum PDC:",
    patient_adherence["pdc"].min()
)

print(
    "Maximum PDC:",
    patient_adherence["pdc"].max()
)

print(
    "Covered days exceeding measurement days:",
    (
        patient_adherence["total_covered_days"]
        > patient_adherence["measurement_days"]
    ).sum()
)

print(
    "Adherent groups:",
    patient_adherence["adherent"].sum()
)

print(
    "Non-adherent groups:",
    (~patient_adherence["adherent"]).sum()
)


PDC VALIDATION SUMMARY
Total patient-measure groups: 3565
Missing PDC values: 0
PDC below zero: 0
PDC above 1: 0
Minimum PDC: 0.1643835616438356
Maximum PDC: 1.0
Covered days exceeding measurement days: 0
Adherent groups: 2768
Non-adherent groups: 797


In [15]:

# ==========================================
# ADHERENCE SUMMARY BY MEASURE
# ==========================================

# Group patients by adherence measure

adherence_summary = (
    patient_adherence
    .groupby("measure", as_index=False)
    .agg(
        eligible_patients=("patient_id", "count"),

        adherent_patients=("adherent", "sum"),

        average_pdc=("pdc", "mean"),

        median_pdc=("pdc", "median")
    )
)

# Calculate non-adherent patients

adherence_summary["non_adherent_patients"] = (
    adherence_summary["eligible_patients"]
    - adherence_summary["adherent_patients"]
)

# Calculate adherence rate

adherence_summary["adherence_rate"] = (
    adherence_summary["adherent_patients"]
    / adherence_summary["eligible_patients"]
) * 100

# Convert average and median PDC to percentages

adherence_summary["average_pdc"] *= 100
adherence_summary["median_pdc"] *= 100

# Round percentages

percentage_columns = [
    "average_pdc",
    "median_pdc",
    "adherence_rate"
]

adherence_summary[percentage_columns] = (
    adherence_summary[percentage_columns].round(2)
)

print("ADHERENCE SUMMARY BY MEASURE")
print("=" * 60)

display(adherence_summary)


ADHERENCE SUMMARY BY MEASURE


,measure,eligible_patients,adherent_patients,average_pdc,median_pdc,non_adherent_patients,adherence_rate
0,cholesterol_statins,1166,910,88.26,100.0,256,78.04
1,diabetes,1207,927,87.29,100.0,280,76.80
2,hypertension_ras,1192,931,88.45,100.0,261,78.10


In [16]:

# Identify the measure with the lowest adherence rate

lowest_measure = adherence_summary.loc[
    adherence_summary["adherence_rate"].idxmin()
]

print("LOWEST ADHERENCE MEASURE")
print("=" * 45)

print("Measure:", lowest_measure["measure"])

print(
    "Adherence rate:",
    str(lowest_measure["adherence_rate"]) + "%"
)

print(
    "Eligible patients:",
    lowest_measure["eligible_patients"]
)


LOWEST ADHERENCE MEASURE
Measure: diabetes
Adherence rate: 76.8%
Eligible patients: 1207


In [18]:
from pathlib import Path

print("Path library imported successfully!")

Path library imported successfully!


In [19]:

# Save patient adherence dataset

output_path = Path("../outputs")
output_path.mkdir(parents=True, exist_ok=True)

patient_adherence.to_csv(
    output_path / "patient_adherence.csv",
    index=False
)

print("patient_adherence.csv exported successfully!")
print("Total patient-measure records:", len(patient_adherence))


patient_adherence.csv exported successfully!
Total patient-measure records: 3565


In [20]:

# ==========================================
# OUTREACH ANALYSIS: SET CUTOFF DATE
# ==========================================

outreach_cutoff = pd.Timestamp("2025-09-30")

# Load patient-measure groups whose measurement
# periods were still active after September 30

outreach_windows = pd.read_sql_query(
    """
    SELECT
        patient_id,
        measure,
        measurement_start,
        measurement_end,
        measurement_days
    FROM patient_measure_cohort_window
    WHERE DATE(measurement_start) <= '2025-09-30'
      AND DATE(measurement_end) > '2025-09-30'
    """,
    con=engine
)

# Convert date columns

for col in ["measurement_start", "measurement_end"]:
    outreach_windows[col] = pd.to_datetime(
        outreach_windows[col],
        errors="raise"
    )

print("OUTREACH WINDOW SUMMARY")
print("=" * 45)

print("Outreach cutoff:", outreach_cutoff.date())

print(
    "Active patient-measure groups:",
    len(outreach_windows)
)

print(
    "Unique patients:",
    outreach_windows["patient_id"].nunique()
)


OUTREACH WINDOW SUMMARY
Outreach cutoff: 2025-09-30
Active patient-measure groups: 3443
Unique patients: 2104


In [21]:

# ==========================================
# LOAD REFILLS AVAILABLE BY THE CUTOFF DATE
# ==========================================

outreach_refills_asof = pd.read_sql_query(
    """
    SELECT
        r.fill_id,
        r.patient_id,
        r.measure,
        r.drug_name,
        r.filled_on,
        r.days_supply,
        r.measurement_start,
        r.measurement_end,
        r.measurement_days

    FROM cohort_refills AS r

    INNER JOIN patient_measure_cohort_window AS w

        ON r.patient_id = w.patient_id
        AND r.measure = w.measure

    WHERE DATE(w.measurement_start) <= '2025-09-30'
      AND DATE(w.measurement_end) > '2025-09-30'
      AND DATE(r.filled_on) <= '2025-09-30'

    ORDER BY
        r.patient_id,
        r.measure,
        r.filled_on,
        r.fill_id
    """,
    con=engine
)

# Convert date columns

for col in [
    "filled_on",
    "measurement_start",
    "measurement_end"
]:
    outreach_refills_asof[col] = pd.to_datetime(
        outreach_refills_asof[col],
        errors="raise"
    )

print("OUTREACH REFILL DATA LOADED")
print("=" * 45)

print("Refill records available by cutoff:",
      len(outreach_refills_asof))

print("Latest refill date used:",
      outreach_refills_asof["filled_on"].max().date())

print(
    "Records after cutoff included:",
    (outreach_refills_asof["filled_on"] > outreach_cutoff).sum()
)


OUTREACH REFILL DATA LOADED
Refill records available by cutoff: 18331
Latest refill date used: 2025-09-30
Records after cutoff included: 0


In [22]:

# ==========================================
# COVERAGE AS OF SEPTEMBER 30, 2025
# ==========================================

# Define the outreach cutoff
outreach_cutoff = pd.Timestamp("2025-09-30")


def calculate_coverage_asof(group):

    group = group.sort_values(
        ["filled_on", "fill_id"]
    ).copy()

    previous_effective_end = None
    covered_days_list = []
    effective_starts = []
    effective_ends = []

    for row in group.itertuples(index=False):

        fill_date = pd.Timestamp(row.filled_on).normalize()
        window_start = pd.Timestamp(row.measurement_start).normalize()
        window_end = pd.Timestamp(row.measurement_end).normalize()

        # Push overlapping refills forward
        if previous_effective_end is None:
            effective_start = fill_date
        else:
            effective_start = max(
                fill_date,
                previous_effective_end + pd.Timedelta(days=1)
            )

        # Preserve the full days supply
        effective_end = (
            effective_start
            + pd.Timedelta(days=int(row.days_supply) - 1)
        )

        # Count coverage only up to September 30
        covered_start = max(effective_start, window_start)

        covered_end = min(
            effective_end,
            window_end,
            outreach_cutoff
        )

        covered_days = max(
            0,
            (covered_end - covered_start).days + 1
        )

        effective_starts.append(effective_start)
        effective_ends.append(effective_end)
        covered_days_list.append(covered_days)

        # Carry forward the remaining supply
        previous_effective_end = effective_end

    group["effective_start"] = effective_starts
    group["effective_end"] = effective_ends
    group["covered_days_asof"] = covered_days_list

    return group


# Process each patient-measure group

outreach_processed_groups = []

for _, group in outreach_refills_asof.groupby(
    ["patient_id", "measure"],
    sort=False
):

    outreach_processed_groups.append(
        calculate_coverage_asof(group)
    )


# Combine processed refill records

outreach_covered_refills = pd.concat(
    outreach_processed_groups,
    ignore_index=True
)

print("OUTREACH COVERAGE CALCULATION COMPLETED!")
print("Refill records:", len(outreach_covered_refills))

print(
    "Patient-measure groups:",
    outreach_covered_refills.groupby(
        ["patient_id", "measure"]
    ).ngroups
)

print(
    "Maximum refill date used:",
    outreach_covered_refills["filled_on"].max().date()
)


OUTREACH COVERAGE CALCULATION COMPLETED!
Refill records: 18331
Patient-measure groups: 3443
Maximum refill date used: 2025-09-30


In [23]:

print("OUTREACH COVERAGE VALIDATION")
print("=" * 50)

print(
    "Negative covered days:",
    (outreach_covered_refills["covered_days_asof"] < 0).sum()
)

print(
    "Coverage exceeding measurement window:",
    (
        outreach_covered_refills["covered_days_asof"]
        > outreach_covered_refills["days_supply"]
    ).sum()
)

print(
    "Refill records after cutoff:",
    (
        outreach_covered_refills["filled_on"]
        > outreach_cutoff
    ).sum()
)

print(
    "Total covered days as of cutoff:",
    outreach_covered_refills["covered_days_asof"].sum()
)


OUTREACH COVERAGE VALIDATION
Negative covered days: 0
Coverage exceeding measurement window: 0
Refill records after cutoff: 0
Total covered days as of cutoff: 712846


In [24]:

# ==========================================
# BUILD OUTREACH ASSESSMENT
# ==========================================

# Aggregate covered days by patient and measure

asof_group_summary = (
    outreach_covered_refills
    .groupby(
        ["patient_id", "measure"],
        as_index=False
    )
    .agg(
        covered_days_asof=("covered_days_asof", "sum"),
        refills_asof=("fill_id", "nunique")
    )
)

# Attach measurement windows

outreach_assessment = outreach_windows.merge(
    asof_group_summary,
    on=["patient_id", "measure"],
    how="left",
    validate="one_to_one"
)

outreach_assessment["covered_days_asof"] = (
    outreach_assessment["covered_days_asof"].fillna(0)
)

# Calculate elapsed and remaining measurement days

outreach_assessment["elapsed_days"] = (
    outreach_cutoff - outreach_assessment["measurement_start"]
).dt.days + 1

outreach_assessment["remaining_days"] = (
    outreach_assessment["measurement_end"] - outreach_cutoff
).dt.days

# Calculate current PDC

outreach_assessment["current_pdc"] = (
    outreach_assessment["covered_days_asof"]
    / outreach_assessment["elapsed_days"]
)

# Calculate maximum achievable PDC

outreach_assessment["maximum_pdc"] = (
    outreach_assessment["covered_days_asof"]
    + outreach_assessment["remaining_days"]
) / outreach_assessment["measurement_days"]

# Calculate additional covered days needed

outreach_assessment["required_covered_days"] = np.ceil(
    0.80 * outreach_assessment["measurement_days"]
).astype(int)

outreach_assessment["additional_days_needed"] = (
    outreach_assessment["required_covered_days"]
    - outreach_assessment["covered_days_asof"]
).clip(lower=0)

# Identify patients currently below 80%
# who can still reach 80%

outreach_assessment["outreach_eligible"] = (
    (outreach_assessment["current_pdc"] < 0.80)
    & (outreach_assessment["maximum_pdc"] >= 0.80)
    & (outreach_assessment["remaining_days"] > 0)
    & (
        outreach_assessment["additional_days_needed"]
        <= outreach_assessment["remaining_days"]
    )
)

print("OUTREACH ELIGIBILITY ASSESSMENT")
print("=" * 50)

print("Active patient-measure groups:", len(outreach_assessment))

print(
    "Currently below 80%:",
    (outreach_assessment["current_pdc"] < 0.80).sum()
)

print(
    "Potentially able to reach 80%:",
    outreach_assessment["outreach_eligible"].sum()
)

print(
    "Unique patients eligible for outreach:",
    outreach_assessment.loc[
        outreach_assessment["outreach_eligible"],
        "patient_id"
    ].nunique()
)


OUTREACH ELIGIBILITY ASSESSMENT
Active patient-measure groups: 3443
Currently below 80%: 643
Potentially able to reach 80%: 106
Unique patients eligible for outreach: 100


In [25]:

# ==========================================
# CREATE FINAL OUTREACH LIST
# ==========================================

# Select only eligible outreach candidates

outreach_list = outreach_assessment.loc[
    outreach_assessment["outreach_eligible"]
].copy()

# Add percentage columns

outreach_list["current_pdc_percentage"] = (
    outreach_list["current_pdc"] * 100
).round(2)

outreach_list["maximum_pdc_percentage"] = (
    outreach_list["maximum_pdc"] * 100
).round(2)

# Add assessment date

outreach_list["assessment_date"] = outreach_cutoff

# Select useful output columns

outreach_columns = [
    "patient_id",
    "measure",
    "assessment_date",
    "measurement_end",
    "covered_days_asof",
    "measurement_days",
    "current_pdc_percentage",
    "maximum_pdc_percentage",
    "additional_days_needed",
    "remaining_days",
    "refills_asof"
]

outreach_list = outreach_list[outreach_columns].copy()

# Prioritize candidates needing fewer additional covered days

outreach_list = outreach_list.sort_values(
    by=["measure", "additional_days_needed"],
    ascending=[True, True]
)

# Validate eligibility conditions

assert (outreach_list["current_pdc_percentage"] < 80).all()

assert (outreach_list["maximum_pdc_percentage"] >= 80).all()

assert (outreach_list["remaining_days"] > 0).all()

# Create output directory

from pathlib import Path

output_path = Path("../outputs")
output_path.mkdir(parents=True, exist_ok=True)

# Export locally

outreach_list.to_csv(
    output_path / "outreach_list.csv",
    index=False
)

print("OUTREACH LIST EXPORTED SUCCESSFULLY!")
print("Patient-measure records:", len(outreach_list))
print("Unique patients:", outreach_list["patient_id"].nunique())
print("Measures represented:", outreach_list["measure"].nunique())
print("Eligibility validation: PASSED")


OUTREACH LIST EXPORTED SUCCESSFULLY!
Patient-measure records: 106
Unique patients: 100
Measures represented: 3
Eligibility validation: PASSED


In [26]:

# ==========================================
# EXPORT ADHERENCE SUMMARY
# ==========================================

adherence_summary = (
    patient_adherence
    .groupby("measure", as_index=False)
    .agg(
        eligible_patient_measure_groups=("patient_id", "size"),
        adherent_groups=("adherent", "sum"),
        average_pdc=("pdc", "mean"),
        median_pdc=("pdc", "median")
    )
)

# Calculate non-adherent groups

adherence_summary["non_adherent_groups"] = (
    adherence_summary["eligible_patient_measure_groups"]
    - adherence_summary["adherent_groups"]
)

# Calculate adherence rate

adherence_summary["adherence_rate"] = (
    adherence_summary["adherent_groups"]
    / adherence_summary["eligible_patient_measure_groups"]
) * 100

# Convert PDC statistics into percentages

adherence_summary["average_pdc"] *= 100
adherence_summary["median_pdc"] *= 100

# Round percentage values

percentage_columns = [
    "average_pdc",
    "median_pdc",
    "adherence_rate"
]

adherence_summary[percentage_columns] = (
    adherence_summary[percentage_columns].round(2)
)

# Export summary

adherence_summary.to_csv(
    output_path / "adherence_summary.csv",
    index=False
)

print("ADHERENCE SUMMARY EXPORTED SUCCESSFULLY!")
print("Measures:", len(adherence_summary))


ADHERENCE SUMMARY EXPORTED SUCCESSFULLY!
Measures: 3


In [27]:

# Verify required output files

required_files = [
    "patient_adherence.csv",
    "adherence_summary.csv",
    "outreach_list.csv"
]

print("FINAL PROJECT OUTPUT VERIFICATION")
print("=" * 45)

for filename in required_files:

    file_path = output_path / filename

    print(
        filename,
        "—",
        "EXISTS" if file_path.exists() else "MISSING"
    )

    if file_path.exists():
        print("File size:", file_path.stat().st_size, "bytes")


FINAL PROJECT OUTPUT VERIFICATION
patient_adherence.csv — EXISTS
File size: 343977 bytes
adherence_summary.csv — EXISTS
File size: 255 bytes
outreach_list.csv — EXISTS
File size: 8008 bytes
